In [ ]:
from pathlib import Path

import imageio.v2 as imageio

station_string = '011'

def pngs_to_video(
    image_folder,
    output_file,
    fps=2,
    pattern="*.png",
    repeats=1,
):
    """
    Create an MP4 video from PNG files in a folder.

    Parameters
    ----------
    image_folder : str or pathlib.Path
        Folder containing PNG images.

    output_file : str or pathlib.Path
        Output video filename, e.g. "stations.mp4".

    fps : int or float
        Frames per second.

    pattern : str
        File pattern used to find images.

    repeats : int
        Number of times to repeat the complete image sequence.
    """

    if repeats < 1 or not isinstance(repeats, int):
        raise ValueError("repeats must be a positive integer.")

    image_folder = Path(image_folder)
    output_file = Path(output_file)

    image_files = sorted(
        path
        for path in image_folder.glob(pattern)
        if not path.name.startswith("._")
    )

    if not image_files:
        raise ValueError(
            f"No images matching {pattern!r} found in {image_folder}"
        )

    with imageio.get_writer(
        output_file,
        fps=fps,
        codec="libx264",
        quality=8,
    ) as writer:

        for _ in range(repeats):
            for image_file in image_files:
                frame = imageio.imread(image_file)
                writer.append_data(frame)

    print(f"Saved video to: {output_file}")

In [ ]:
input_folder = '/Volumes/Extreme SSD/Bloom 202506/8D_rgbs'
output_folder = '/Volumes/Extreme SSD/Bloom 202506/8D_rgbs/vid.mp4'
pngs_to_video(input_folder, output_folder, fps = 1, repeats = 5)


In [ ]:
from pathlib import Path
import re

from PIL import Image


rgb_folder = Path(
    "/Volumes/Extreme SSD/Bloom 202506/8D_rgbs"
)

graph_folder = Path(
    "/Volumes/Extreme SSD/Bloom 202506/timeseries/"
    f"SS20250602_{station_string}/stations_isolated"
)

output_folder = Path(
    "/Volumes/Extreme SSD/Bloom 202506/timeseries/"
    f"SS20250602_{station_string}/frames"
)

output_folder.mkdir(parents=True, exist_ok=True)

station = f"SS20250602_{station_string}"

rgb_files = sorted(
    path
    for path in rgb_folder.glob("*.png")
    if not path.name.startswith("._")
)

for rgb_path in rgb_files:

    # Search only the filename, not the full path
    match = re.search(
        r"\d{4}-\d{2}-\d{2}_\d{4}-\d{2}-\d{2}",
        rgb_path.name,
    )

    if match is None:
        print(f"Skipping {rgb_path.name}: no date range found")
        continue

    date_range = match.group(0)

    graph_path = (
        graph_folder
        / f"{station}_{date_range}_rrs_spectrum.png"
    )

    if not graph_path.exists():
        print(f"Skipping {rgb_path.name}: graph not found")
        print(f"Expected: {graph_path}")
        continue

    output_path = output_folder / rgb_path.name

    with Image.open(rgb_path) as rgb_image, Image.open(graph_path) as graph_image:

        rgb_image = rgb_image.convert("RGB")
        graph_image = graph_image.convert("RGB")

        # Resize graph to RGB height while preserving aspect ratio
        graph_height = rgb_image.height
        graph_width = round(
            graph_image.width
            * graph_height
            / graph_image.height
        )

        graph_image = graph_image.resize(
            (graph_width, graph_height),
            Image.Resampling.LANCZOS,
        )

        # Create side-by-side image
        combined = Image.new(
            "RGB",
            (
                rgb_image.width + graph_image.width,
                rgb_image.height,
            ),
            color="white",
        )

        combined.paste(rgb_image, (0, 0))
        combined.paste(
            graph_image,
            (rgb_image.width, 0),
        )

        combined.save(output_path)

    print(f"Saved: {output_path}")

In [ ]:
image_folder = f'/Volumes/Extreme SSD/Bloom 202506/timeseries/SS20250602_{station_string}/frames'
output_file = f'/Volumes/Extreme SSD/Bloom 202506/timeseries/SS20250602_{station_string}/{station_string}_rgb_graphs.mp4'

pngs_to_video(
    image_folder,
    output_file,
    fps=1,
    pattern="*.png",
    repeats=5,
)